# Dijet eta-dependent smeared-Reco to Gen closures

Compare the default eta-dependent JER-smeared reconstructed dijet pseudorapidity distribution with nominal Gen. For every configured $p_T^{ave}$ interval, the notebook writes a full $\eta_{CM}^{dijet}$ distribution closure and an unnormalized Forward/Backward-ratio closure. The lower panel of each canvas shows the smeared-Reco result divided by Gen.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.plotting import draw_closure


In [ ]:
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

`hRecoDijetPtEta*JerDefExtra` is the default JER-smearing variation with eta-dependent scaling. The runnable default uses the current p-going output; select `Pbgoing` for the opposite orientation. Select `combined` only after its merged file has been regenerated with the `JerDefExtra` histograms. The standard eta-cut index corresponds to $|\eta_{CM}^{jet}|<1.9$, and all configured half-open $p_T^{ave}$ intervals are processed. Full distributions are unit-normalized by bin content; Forward and Backward inputs are not normalized before division. Independent ROOT error propagation is used because these weighted samples are not binomial subset efficiencies.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'          # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BINS = tuple(DIJET_PTAVE_BINS)
REBIN_ETA = 2
NORMALIZATION = 'integral'   # none, integral, or bin_width
FORWARD_BACKWARD_RATIO_OPTION = ''  # F / B: never use binomial errors
FULL_CLOSURE_RATIO_OPTION = ''      # use 'B' only for a true subset ratio
FB_CLOSURE_RATIO_OPTION = ''        # user choice for (F/B)_a / (F/B)_b: '' or 'B'
FULL_RATIO_RANGE = (0.75, 1.25)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.75, 1.25)
SAVE_PNG = False
DRAW_GRID = True
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_CLOSURE_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output'
    / 'dijet_reco_smeared_to_gen_closures',
))

CURVES = (
    DijetClosureCurve(
        'Reco smeared (JER default, #eta-dep.)',
        'hRecoDijetPtEtaCMJerDefExtra_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJerDefExtra_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJerDefExtra_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
        'hGenDijetPtEtaForward_{eta_cut_index}',
        'hGenDijetPtEtaBackward_{eta_cut_index}',
    ),
)
NOMINAL = 'Gen'
STYLE_INDICES = {
    'Reco smeared (JER default, #eta-dep.)': 0,
    'Gen': 1,
}

if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid eta-cut index: {ETA_CUT_INDEX}')
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]

In [ ]:
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE

## Full distributions and Forward/Backward ratios

Each full-distribution PDF overlays unit-normalized smeared Reco and Gen, with smeared Reco / Gen below. Each F/B PDF overlays the ratios formed from the original projected yields, with $(F/B)_{reco\ smeared}/(F/B)_{gen}$ below.

In [ ]:
closure_results = {}
eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_x_range = (0.0, ETA_CUT + 0.1)
eta_cut_tag = int(round(10.0 * ETA_CUT))

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    selection_tag = (
        f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
    )
    eta_shapes, fb_ratios, keys = build_dijet_gen_comparisons(
        INPUT_FILE, CURVES, eta_cut_index=ETA_CUT_INDEX,
        ptave_range=ptave_range, nominal=NOMINAL,
        rebin_eta=REBIN_ETA, normalization=NORMALIZATION,
        ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
    )
    annotations = (
        GENERATOR.capitalize(),
        'Gen and eta-dependent JER-smeared Reco dijets',
        'CM frame',
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} > 50 GeV',
        'p_{T}^{SubLead} > 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    eta_y_title = {
        'none': 'Dijets / bin',
        'integral': 'Fraction of dijets / bin',
        'bin_width': '1/N dN/d#eta_{CM}^{dijet}',
    }[NORMALIZATION]
    full_tag = f'{GENERATOR}_{DIRECTION}_recoSmearedToGen_full_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
    fb_tag = f'{GENERATOR}_{DIRECTION}_recoSmearedToGen_fb_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'

    eta_canvas, eta_to_gen = draw_closure(
        eta_shapes, NOMINAL, title='',
        x_title='#eta_{CM}^{dijet}', y_title=eta_y_title,
        ratio_range=FULL_RATIO_RANGE, x_range=eta_x_range,
        annotations=annotations, grid=DRAW_GRID, headroom=1.6,
        draw_nominal_ratio=False, ratio_option=FULL_CLOSURE_RATIO_OPTION,
        style_indices=STYLE_INDICES,
        output=OUTPUT_DIR / f'{full_tag}.pdf', save_png=SAVE_PNG,
        canvas_name=full_tag,
    )
    fb_canvas, fb_to_gen = draw_closure(
        fb_ratios, NOMINAL, title='',
        x_title='#eta_{CM}^{dijet}', y_title='Forward / Backward',
        ratio_range=FB_DOUBLE_RATIO_RANGE, x_range=fb_x_range,
        y_range=FB_RANGE, annotations=annotations, grid=DRAW_GRID,
        headroom=1.6, draw_nominal_ratio=False,
        ratio_option=FB_CLOSURE_RATIO_OPTION,
        style_indices=STYLE_INDICES,
        output=OUTPUT_DIR / f'{fb_tag}.pdf', save_png=SAVE_PNG,
        canvas_name=fb_tag,
    )
    closure_results[ptave_range] = {
        'eta_shapes': eta_shapes, 'eta_to_gen': eta_to_gen,
        'forward_backward': fb_ratios,
        'forward_backward_to_gen': fb_to_gen,
        'eta_canvas': eta_canvas,
        'forward_backward_canvas': fb_canvas, 'keys': keys,
    }
    print(selection_tag, keys)
    display(eta_canvas)
    display(fb_canvas)

## Numerical audit

Report both full-distribution integral conventions and the finite nonzero extrema of the smeared-Reco / Gen closure ratios.

In [ ]:
def finite_nonzero_range(histogram):
    values = [
        histogram.GetBinContent(index)
        for index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(index) != 0.0
        and math.isfinite(histogram.GetBinContent(index))
    ]
    return (min(values), max(values)) if values else None

reco_label = CURVES[0].label
for ptave_range, result in closure_results.items():
    reco_shape = result['eta_shapes'][reco_label]
    gen_shape = result['eta_shapes'][NOMINAL]
    print(f'\npTave interval {ptave_range}, eta cut {ETA_CUT:g}')
    print({
        'reco bin sum': reco_shape.Integral(),
        'gen bin sum': gen_shape.Integral(),
        'reco width integral': reco_shape.Integral('width'),
        'gen width integral': gen_shape.Integral('width'),
        'reco/gen range': finite_nonzero_range(
            result['eta_to_gen'][reco_label]),
        '(F/B)_reco/(F/B)_gen range': finite_nonzero_range(
            result['forward_backward_to_gen'][reco_label]),
    })